# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [1]:
class matrix:
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            n, m = args
            self.__data = [[0.0 for _ in range(m)] for _ in range(n)]
        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]
            if not data or not isinstance(data[0], list):
                raise ValueError("Must provide a list of lists.")
            cols = len(data[0])
            for row in data:
                if len(row) != cols:
                    raise ValueError("All rows must have the same number of columns.")
            self.__data = [[val for val in row] for row in data]
        else:
            raise ValueError("Invalid initialization arguments.")
            
    def assign(self, other):
        target_n, target_m = self.shape()
        
        if isinstance(other, matrix):
            if other.shape() != (target_n, target_m):
                raise ValueError("Size mismatch for assignment.")
            self.__data = [[val for val in row] for row in other.to_list()]
        elif isinstance(other, list):
            temp_mat = matrix(other)
            if temp_mat.shape() != (target_n, target_m):
                raise ValueError("Size mismatch for assignment.")
            self.__data = temp_mat.to_list()
        else:
            raise ValueError("Unsupported type for assignment.")

    def __getitem__(self, key):
        if isinstance(key, tuple):
            row_idx, col_idx = key
            if isinstance(row_idx, slice) or isinstance(col_idx, slice):
                rows = self.__data[row_idx] if isinstance(row_idx, slice) else [self.__data[row_idx]]
                sliced_data = []
                for row in rows:
                    sliced_data.append(row[col_idx] if isinstance(col_idx, slice) else [row[col_idx]])
                return matrix(sliced_data)
            return self.__data[row_idx][col_idx]
        elif isinstance(key, slice):
            return matrix(self.__data[key])
        else:
            return self.__data[key]
            
    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            row_idx, col_idx = key
            self.__data[row_idx][col_idx] = value
        else:
            self.__data[key] = value

    def shape(self):
        n = len(self.__data)
        m = len(self.__data[0]) if n > 0 else 0
        return (n, m)

    def transpose(self):
        n, m = self.shape()
        transposed = [[self.__data[i][j] for i in range(n)] for j in range(m)]
        return matrix(transposed)

    def row(self, n):
        return matrix([self.__data[n]])

    def column(self, n):
        col_data = [[self.__data[i][n]] for i in range(self.shape()[0])]
        return matrix(col_data)

    def to_list(self):
        return [[val for val in row] for row in self.__data]

    def block(self, n_0, n_1, m_0, m_1):
        sliced_data = [r[n_0:n_1] for r in self.__data[m_0:m_1]]
        return matrix(sliced_data)

    def scalarmul(self, c):
        n, m = self.shape()
        new_data = [[self.__data[i][j] * c for j in range(m)] for i in range(n)]
        return matrix(new_data)

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must be the same size to add.")
        n, m = self.shape()
        N_list = N.to_list()
        new_data = [[self.__data[i][j] + N_list[i][j] for j in range(m)] for i in range(n)]
        return matrix(new_data)

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must be the same size to subtract.")
        n, m = self.shape()
        N_list = N.to_list()
        new_data = [[self.__data[i][j] - N_list[i][j] for j in range(m)] for i in range(n)]
        return matrix(new_data)

    def mat_mult(self, N):
        n1, m1 = self.shape()
        n2, m2 = N.shape()
        if m1 != n2:
            raise ValueError("Incompatible dimensions for matrix multiplication.")
        
        N_list = N.to_list()
        new_data = [[0.0 for _ in range(m2)] for _ in range(n1)]
        for i in range(n1):
            for j in range(m2):
                new_data[i][j] = sum(self.__data[i][k] * N_list[k][j] for k in range(m1))
        return matrix(new_data)

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must be the same size for element-wise multiplication.")
        n, m = self.shape()
        N_list = N.to_list()
        new_data = [[self.__data[i][j] * N_list[i][j] for j in range(m)] for i in range(n)]
        return matrix(new_data)

    def equals(self, N):
        if not isinstance(N, matrix):
            return False
        return self.__data == N.to_list()
        
    def __add__(self, other):
        return self.add(other)
        
    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, matrix):
            return self.mat_mult(other)
        raise ValueError("Unsupported type for multiplication.")

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        raise ValueError("Unsupported type for multiplication.")

    def __eq__(self, other):
        return self.equals(other)

    def __repr__(self):
        return "\n".join(["\t".join(map(str, row)) for row in self.__data])

def constant(n, m, c):
    return matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    mat = zeros(n, n)
    for i in range(n):
        mat[i, i] = 1.0
    return mat



In [4]:
m1 = matrix(2, 3)
print(m1.shape())
print(m1.to_list())

m2 = matrix([[1, 2, 3], [4, 5, 6]])
print(m2.shape())
print(m2.to_list())

m1.assign(m2)
print(m1.to_list())

m1.assign([[7, 8, 9], [10, 11, 12]])
print(m1.to_list())

print(m2[0][1])
print(m2[1, 2])

m2[0, 0] = 99
print(m2[0, 0])
m2[0][0] = 1

sliced = m2[0:2, 1:3]
print(sliced.to_list())

m3 = m2.transpose()
print(m3.to_list())

print(m2.row(1).to_list())
print(m2.column(1).to_list())

m4 = matrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
print(m4.block(1, 3, 0, 2).to_list())

A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

print(A.scalarmul(2).to_list())
print((A * 2).to_list())
print((2 * A).to_list())

print(A.add(B).to_list())
print((A + B).to_list())

print(A.sub(B).to_list())
print((A - B).to_list())

print(A.mat_mult(B).to_list())
print((A * B).to_list())

print(A.element_mult(B).to_list())

print(A.equals(matrix([[1, 2], [3, 4]])))
print(A == matrix([[1, 2], [3, 4]]))
print(A == B)

print(constant(2, 2, 7.5).to_list())
print(zeros(2, 2).to_list())
print(ones(2, 2).to_list())
print(eye(2).to_list())

(2, 3)
[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
(2, 3)
[[1, 2, 3], [4, 5, 6]]
[[1, 2, 3], [4, 5, 6]]
[[7, 8, 9], [10, 11, 12]]
2
6
99
[[2, 3], [5, 6]]
[[1, 4], [2, 5], [3, 6]]
[[4, 5, 6]]
[[2], [5]]
[[2, 3], [6, 7]]
[[2, 4], [6, 8]]
[[2, 4], [6, 8]]
[[2, 4], [6, 8]]
[[6, 8], [10, 12]]
[[6, 8], [10, 12]]
[[-4, -4], [-4, -4]]
[[-4, -4], [-4, -4]]
[[19, 22], [43, 50]]
[[19, 22], [43, 50]]
[[5, 12], [21, 32]]
True
True
False
[[7.5, 7.5], [7.5, 7.5]]
[[0.0, 0.0], [0.0, 0.0]]
[[1.0, 1.0], [1.0, 1.0]]
[[1.0, 0.0], [0.0, 1.0]]


In [3]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])
C = matrix([[9, 10], [11, 12]])
I = eye(2)

LHS_1 = (A * B) * C
RHS_1 = A * (B * C)
print("(AB)C == A(BC):", LHS_1 == RHS_1)


LHS_2 = A * (B + C)
RHS_2 = (A * B) + (A * C)
print("A(B+C) == AB + AC:", LHS_2 == RHS_2)


LHS_3 = A * B
RHS_3 = B * A
print("AB != BA:", not (LHS_3 == RHS_3))


LHS_4 = A * I
print("AI == A:", LHS_4 == A)

(AB)C == A(BC): True
A(B+C) == AB + AC: True
AB != BA: True
AI == A: True
